# 동적 웹페이지 크롤링  : (3) Load More
- https://webscraper.io/test-sites/e-commerce/more/computers/laptops

## [문법 설명] Lambda

### 기본 구조
- 한 줄로 간결하게 함수 정의

1. Lamdba 문법의 기본 구조

```python
lambda [parameters] : expression
```

> `주의 사항`
> 1. `lambda` 내부에는 단 하나의 표현식(expression)만 올 수 있음
> 2. `return`, `raise`와 같은 문장(statement)이나 변수 할당문(=)은 사용할 수 없음
> 3. 여러 줄의 코드나 조건문 블록은 작성할 수 없음 -> 단일 삼항 연산자 형태만 허용

2. 매개변수(parameter) 형태
```python
lambda: 'Hi~'          ## 매개변수 없음
lambda x, y: x + y     ## 기본 매개변수
lambda x, y=10: x * y  ## 기본값 설정
lambda *args: sum(args)
lambda **kwargs: kwargs.get('a', 0)
```

### 활용

#### map() : 데이터 변환

##### 일반 함수

In [ ]:
numbers = [1, 3, 4, 6]

## 1. 함수 정의
def square(x):
    print(f'{x = }')
    return x ** 2

## 2. map()에 square()함수 전달
squared = list(map(square, numbers))
squared

##### lambda

In [ ]:
list(map(lambda x: x ** 2, numbers))

#### filter() : 조건 필터링

##### 일반 함수

In [ ]:
numbers = [1, 3, 4, 6]

## 1. 짝수 여부 판별하는 함수 정의 (True/False 반환)
def is_even(x):
    return x % 2 == 0

## 2. filter() 함수에 is_even 함수 전달
list(filter(is_even, numbers))

##### lambda

In [ ]:
list(filter(lambda x: x % 2 == 0, numbers))

#### sorted()

##### 일반 함수

In [ ]:
students = [('홍길동', 85), ('박보검', 100), ('이미자', 70)]

## 1. 정렬 기준이 될 값을 추출하여 리턴
def get_score(student):
    print(f'{student = }')
    return student[1]

## 2. sorted() 함수에 정렬 기준 함수 전달
sorted(students, key=get_score, reverse=True)


##### lambda

In [ ]:
sorted(students, key=lambda student: student[1], reverse=True)

# 라이브러리 불러오기

In [1]:
from datetime import datetime
from pathlib import Path

import pandas as pd

from selenium import webdriver
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# 기본 설정

In [2]:
## URL
TARGET_URL = 'https://webscraper.io/test-sites/e-commerce/more/computers/laptops'

## More 버튼 최대 클릭 횟수
MAX_CLICKS = 3

## 동적 요소 최대 대기 시간(초)
WAIT_TIMEOUT = 10

## 브라우저 화면 표시 여부
## False : 브라우저 화면 표시
## True : 브라우저 화면을 표시하지 않고 실행
HEADLESS = False

## csv 저장 폴더
PROJECT_DIR = Path.cwd().resolve().parents[1]
OUTPUT_DIR = PROJECT_DIR / 'data' / 'dynamic'

## 상품 영역 CSS 선택자
PRODUCT_SELECTOR = 'div.product-wrapper'

## More 버튼 선택자
LOAD_MORE_SELECTOR = 'a.ecomerce-items-scroll-more'

# Chrome WebDriver 생성

In [3]:
def create_driver(headless: bool = HEADLESS) -> webdriver.Chrome:
    """
    Chrome WebDriver를 생성하여 반환한다.

    Args:
        headless:
            True이면 브라우저 화면을 표시하지 않고 실행한다.

    Returns:
        Chrome WebDriver 객체    
    """
    options = Options()

    if headless:
        options.add_argument('--headless=new')

    ## 창 최대화 옵션 추가
    options.add_argument('--start-maximized')

    return webdriver.Chrome(options=options)
    

# Load More 페이지 접속

In [ ]:
driver = create_driver()

In [ ]:
driver.get(TARGET_URL)

In [ ]:
wait = WebDriverWait(driver, WAIT_TIMEOUT)

## 초기 상품 요소가 생성될 때까지 대기
wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, PRODUCT_SELECTOR))
)

print(f'페이지 제목 : {driver.title}')
print(f'현재 URL : {driver.current_url}')

# 초기 상품 개수 확인

In [ ]:
product_elements = driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR)
print(f'초기 상품 수 : {len(product_elements)}')

# 상품 한 건의 데이터 추출

In [ ]:
first_product = product_elements[0]
first_product

In [ ]:
## 상품명과 상세 URL이 있는 a 요소
title_element = first_product.find_element(By.CSS_SELECTOR, 'a.title')
title_element

In [ ]:
## 상품명
title = title_element.get_attribute('title')
title

In [ ]:
## 상세 URL
detail_url = title_element.get_attribute('href')
detail_url

In [ ]:
## 가격
price_text = first_product.find_element(By.CSS_SELECTOR, 'h4.price').text
price_text

In [ ]:
## 설명
description = first_product.find_element(By.CSS_SELECTOR, 'p.description').text
description

In [ ]:
print(f'상품명: {title}')
print(f'상세 URL: {detail_url}')
print(f'가격: {price_text}')
print(f'설명: {description}')

# 상품 요소 한 건을 딕셔너리로 변환

In [4]:
def parse_product_element(product_element: WebElement) -> dict[str: str]:
    """
    상품 WebElement 한 건에서 상품 정보를 추출한다.

    Args:
        product_element:
            상품 영역 WebElement

    Returns:
        상품 정보 딕셔너리
    """
    title_element = product_element.find_element(By.CSS_SELECTOR, 'a.title')
    title = title_element.get_attribute('title')
    detail_url = title_element.get_attribute('href')
    price_text = product_element.find_element(By.CSS_SELECTOR, 'h4.price').text
    description = product_element.find_element(By.CSS_SELECTOR, 'p.description').text

    return {
        'title': title,
        'detail_url' : detail_url,
        'price_text': price_text,
        'description': description,
    }

In [ ]:
first_product_data = parse_product_element(first_product)
first_product_data

# More 버튼 한 번 클릭

버튼을 클릭하기 전에 현재 상품 개수를 저장한다.

```python
before_count
```

버튼 클릭 후에는 상품 개수가 증가할 때까지 기다린다.

In [ ]:
before_count = len(driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR))
before_count

In [ ]:
load_more_button = driver.find_element(By.CSS_SELECTOR, LOAD_MORE_SELECTOR)
load_more_button.text

## 버튼이 화면에 보이지 않는 상태에서 클릭!

In [ ]:
## More 버튼 클릭
# load_more_button.click()

## 버튼이 보이도록 스크롤

In [ ]:
driver.execute_script('arguments[0].scrollIntoView({block: "center"})', load_more_button)

In [ ]:
# driver.execute_script('arguments[0].scrollIntoView({block: "end"})', load_more_button)

In [ ]:
## More 버튼 클릭
load_more_button.click()

In [ ]:
## 상품 개수가 증가할 때까지 대기
wait.until(
    lambda current_driver: 
    len(current_driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR)) 
    > before_count
)

In [ ]:
after_count = len(driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR))
after_count

In [ ]:
print(f'버튼 클릭 전 상품 수 : {before_count}')
print(f'버튼 클릭 후 상품 수 : {after_count}')
print(f'추가된 상품 수 : {after_count - before_count}')

# 브라우저 종료

In [ ]:
driver.quit()
print('브라우저 종료! 🚀')

# [함수 정의] More 버튼 반복 클릭

지금까지 작업을 함수로 정리

```plaintext
페이지 접속
-> 초기 상품 로딩 대기
-> More 버튼 찾기        ---┐
-> 현재 상품 개수 저장      |
-> 버튼 위치로 스크롤       |  반복
-> 버튼 클릭                | 
-> 상품 개수 증가 대기   __」
-> 지정한 횟수만큼 반복        
-> 전체 상품 추출
-> 브라우저 종료
```

In [5]:
def find_load_more_button(driver) -> WebElement | None:
    try:
        return driver.find_element(By.CSS_SELECTOR, LOAD_MORE_SELECTOR)
    except NoSuchElementException:
        return None

In [6]:
def crawl_load_more_products(
    target_url: str = TARGET_URL,
    max_clicks: int = MAX_CLICKS,
    wait_timeout: int = WAIT_TIMEOUT, 
    headless: bool = HEADLESS,
) -> list[dict[str, str]]:
    """
    Load More 테스트 페이지에서
    버튼을 반복 클릭한 후 전체 상품을 수집한다.

    Args:
        target_url:
            크롤링 대상 URL

        max_clicks:
            More 버튼 최대 클릭 횟수

        wait_timeout:
            동적 요소 최대 대기 시간(초)

        headless:
            브라우저 화면 표시 여부
            
    Returns:
        전체 상품 정보 딕셔너리 목록
    """

    driver = create_driver(headless=headless)
    wait = WebDriverWait(driver, wait_timeout)

    try:
        driver.get(target_url)

        ## 초기 상품 목록 대기
        wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, PRODUCT_SELECTOR))
        )
        
        initial_count = len(driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR))
        print(f'초기 상품 수 : {initial_count}')

        for click_count in range(1, max_clicks + 1):
            ## More 버튼 찾기
            load_more_button = find_load_more_button(driver)

            ## 버튼이 없으면, 모든 상품이 로드된 것으로 판단하고 종료
            ## --> 더 이상 표시할 상품이 없다!
            if load_more_button is None:
                print('More 버튼이 없어 추가 로딩을 종료합니다.')
                break

            ## 클릭 전 상품 개수
            before_count = len(driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR))

            ## 버튼 위치로 스크롤
            driver.execute_script(
                'arguments[0].scrollIntoView({block: "center"})', 
                load_more_button
            )

            ## More 버튼 클릭
            load_more_button.click()

            ## 상품 개수가 증가할 때까지 대기
            wait.until(
                lambda current_driver: 
                len(current_driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR)) 
                > before_count
            )

            ## 클릭 후 상품 개수
            after_count = len(driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR))
            print(f'{click_count}회 클릭 완료 : {before_count} -> {after_count}')
        

        ## 모든 동적 로딩이 끝난 후, 현재 화면의 전체 상품 요소 수집
        product_elements = driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR)

        products = [
            parse_product_element(product_element) 
            for product_element 
            in product_elements
        ]

        return products

    finally:
        ## 오류 발생 여부와 관계없이 브라우저 종료
        driver.quit()    

# Load More 크롤링 실행

In [8]:
try:
    products = crawl_load_more_products(headless=True)
except TimeoutException as error:
    print('추가 상품 로딩을 기다리는 중 시간 초과가 발생했습니다.')
    print(f'오류 내용 : {error}')
    raise  ## except에서 현재 처리 중인 예외를 그대로 다시 발생

print()
print(f'전체 수집 상품 수 : {len(products)}')

초기 상품 수 : 6
1회 클릭 완료 : 6 -> 12
2회 클릭 완료 : 12 -> 18
3회 클릭 완료 : 18 -> 24

전체 수집 상품 수 : 24
